# 02 — Timeline des rubriques de l'ordre du jour

Objectif : une timeline interactive des sujets abordés en séance (une pour la Chambre des députés, une pour le Sénat), avec pour chaque rubrique les orateurs qui y prennent la parole.

Règles :
- une **rubrique** est ouverte par un intertitre : entité `TITL` (règle de `debats.section_title`) **ou** bloc `Section-Header` sans `TITL` (faux négatifs du NER) ; la liste est plate, la dernière `TITL` est seulement gardée comme contexte (`parent`) des `Section-Header` qui la suivent ;
- seuls les intertitres **de la séance** comptent : un fascicule contient l'en-tête et le sommaire, une ou plusieurs séances, puis des annexes (questions écrites, réponses des ministres, scrutins…). On suit donc l'état du fascicule :
  - `« 13e SÉANCE »` ouvre une séance (en-tête + sommaire), `« PRÉSIDENCE DE M. … »` ouvre les débats ;
  - un intertitre d'annexe (`QUESTIONS ÉCRITES`, `RÉPONSES DES MINISTRES`, `ANNEXE…`, `Ordre du jour du …`) ou « la séance est levée » ferme les débats ;
  - si `M. le président` reprend la parole en annexe, les débats reprennent (questions écrites insérées en cours de séance, communication tardive) sous le dernier intertitre vu ; si cet intertitre est loin (pages perdues par l'OCR), c'est une rubrique non détectée, et une nouvelle séance si la précédente avait été levée ;
- les numéros de séance imprimés étant parfois mal lus (« 1er SÉANCE » pour « 16e »), une séance est identifiée par sa date et son rang dans le fascicule ;
- certains intertitres ne sont pas des sujets et ne coupent rien : articles d'un texte de loi, listes de scrutin, errata, suspension de séance, titres courants ;
- les **orateurs** sont les `SPK` qui ouvrent une prise de parole (`debats.speaker_cut`), comptés par rubrique.

In [ ]:
import re
import unicodedata
from collections import Counter
from dataclasses import dataclass, field

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from debats import (
    CHAMBER_NAMES,
    OUTPUT_DIR,
    SECTION_LABEL,
    SKIPPED_LABELS,
    Block,
    Session,
    iter_sessions,
    load_session,
    section_title,
    session_paths,
    speaker_cut,
)

## Repérage des intertitres

In [ ]:
# « 13e SÉANCE », « COMPTE RENDU IN EXTENSO — 1re SEANCE » (les « 2e séance du … » en minuscules sont des rappels).
SITTING_RE = re.compile(r"\b(\d+)\s*(?:e|re|er|ème)\s+S[ÉE]ANCE\b")
CHAIR_RE = re.compile(r"^PR[ÉE]SIDENCE DE M", re.IGNORECASE)
CLOSING_RE = re.compile(r"s[ée]ance est lev[ée]e", re.IGNORECASE)
# « M. le président. » ouvrant une intervention (et non « M. le président : « … » » cité dans une réponse).
CHAIR_SPEAKS_RE = re.compile(r"^(?:\s|</?b>)*\.")
ANNEX_RE = re.compile(
    r"QUESTIONS [ÉE]CRITES|R[ÉE]PONSES DES MINISTRES|^ANNEXES?\b|^Ordre du jour du|^Liste des questions",
    re.IGNORECASE,
)
# Intertitres qui ne sont pas des sujets : ils ne coupent pas la rubrique en cours.
NOISE_RE = re.compile(
    # structure des textes de loi
    r"^[\[«\s]*(?:Articles?\s+(?:\d+|unique|1er|premier|additionnel)|TITRE\b|CHAPITRE\b|§|\d+(?:re|e) SECTION|Signé|— \d+ —$)"
    # listes de scrutin, errata
    r"|^(?:Ont vot[ée]|N'ont|N'a pas|Absents?|Ne peu|Errat|Rectification)"
    # suspension, titres courants
    r"|^(?:SUSPENSION DE LA S[ÉE]ANCE|JOURNAL OFFICIEL|D[ÉE]BATS PARLEMENTAIRES|CHAMBRE DES D[ÉE]PUT[ÉE]S$|S[ÉE]NAT$|SOMMAIRE$)"
    r"|IN EXTENSO|L[ÉE]GISLATURE",
    re.IGNORECASE,
)


def clean(text: str) -> str:
    return " ".join(text.split())


def heading(block: Block) -> tuple[str, str] | None:
    # (titre, source) si le bloc est un intertitre ; source = « TITL » ou « Section-Header » (faux négatif du NER).
    if (title := section_title(block)) is not None:
        return title, "TITL"
    if block.label == SECTION_LABEL and (text := clean(block.text)):
        return text, SECTION_LABEL
    return None

## Découpage en rubriques

États du fascicule : `front` (en-tête, sommaire), `debate`, `annex` (annexes, reprise possible), `closed` (après « la séance est levée »).

In [ ]:
UNKNOWN_TITLE = "(rubrique non détectée)"


@dataclass
class Rubric:
    sitting: int  # rang de la séance dans le fascicule (1, 2…)
    number: int | None  # numéro de séance imprimé (OCR, None si l'en-tête manque)
    rank: int  # position dans la séance
    title: str
    source: str  # TITL, Section-Header ou « - » (reprise sans intertitre)
    parent: str | None  # dernière TITL, pour un Section-Header
    page: int
    speakers: Counter = field(default_factory=Counter)


def split_rubrics(session: Session) -> list[Rubric]:
    rubrics: list[Rubric] = []
    state = "front"
    sitting, number = 0, None
    parent: str | None = None
    pending: tuple[str, str, int] | None = None  # dernier intertitre vu en annexe
    current: Rubric | None = None

    def open_rubric(title: str, source: str, page: int) -> None:
        nonlocal current
        rank = sum(r.sitting == sitting for r in rubrics)
        current = Rubric(sitting, number, rank, title, source, None if source == "TITL" else parent, page)
        rubrics.append(current)

    for block in session.blocks:
        if block.label in SKIPPED_LABELS:
            continue
        text = clean(block.text)

        if (m := SITTING_RE.search(text)) and (block.label == SECTION_LABEL or block.entities_of("TITL")):
            state, sitting, number, parent, current = "front", sitting + 1, int(m.group(1)), None, None
            continue
        if CHAIR_RE.match(text):  # aussi aux changements de président en cours de séance
            if state == "front":
                state = "debate"
            continue

        if (h := heading(block)) is not None:
            title, source = h
            if NOISE_RE.search(title):
                continue
            if ANNEX_RE.search(title):
                if state == "debate":
                    state = "annex"
                current, pending = None, None
            elif state in ("annex", "closed"):
                pending = (title, source, block.page)
            elif state == "debate":
                if source == "TITL":
                    parent = title
                open_rubric(title, source, block.page)
            continue

        for spk in block.entities_of("SPK"):
            if speaker_cut(block, spk) is None:
                continue
            name = clean(spk.text)
            if state in ("annex", "closed") and name == "président" and CHAIR_SPEAKS_RE.match(block.html[spk.end :]):
                if pending is not None and pending[2] >= block.page - 1:
                    open_rubric(*pending)
                else:
                    # Pages perdues : le dernier intertitre vu est celui des annexes. Après la levée de
                    # séance, c'est la séance suivante dont l'en-tête manque.
                    if state == "closed":
                        sitting, number = sitting + 1, None
                    open_rubric(UNKNOWN_TITLE, "-", block.page)
                state = "debate"
            if state == "debate" and current is not None:
                current.speakers[name] += 1

        if state == "debate" and CLOSING_RE.search(text):
            state, current = "closed", None
    return rubrics

## Une séance

In [ ]:
session = load_session(session_paths()[1])
pd.DataFrame(
    {
        "séance": r.sitting,
        "n°": r.number,
        "page": r.page,
        "source": r.source,
        "rubrique": r.title,
        "parent": r.parent,
        "interventions": r.speakers.total(),
        "orateurs": ", ".join(name for name, _ in r.speakers.most_common()),
    }
    for r in split_rubrics(session)
)

## Toutes les séances

In [ ]:
rows = []
for session in iter_sessions():
    for r in split_rubrics(session):
        rows.append(
            {
                "fichier": session.name,
                "chambre": session.chamber,
                "date": pd.Timestamp(session.date),
                "séance": r.sitting,
                "n_séance": r.number,
                "rang": r.rank,
                "page": r.page,
                "source": r.source,
                "rubrique": r.title,
                "parent": r.parent,
                "interventions": r.speakers.total(),
                "nb_orateurs": len(r.speakers),
                "orateurs": r.speakers.most_common(),
            }
        )
rubrics = pd.DataFrame(rows)
rubrics.groupby("chambre").agg(
    fichiers=("fichier", "nunique"),
    rubriques=("rubrique", "size"),
    dont_section_header=("source", lambda s: (s == SECTION_LABEL).sum()),
    sans_orateur=("interventions", lambda s: (s == 0).sum()),
)

## Arbre des rubriques

Tous les titres de la période, par chambre, à lire en regard de la timeline : une rubrique `TITL` (niveau 1) et, en dessous, les `Section-Header` qui la suivent (niveau 2). Les titres identiques à la casse, aux accents et à la ponctuation finale près sont regroupés (« Colonies. » = « COLONIES ») ; l'ordre est celui de la première apparition. Les rubriques de procédure (procès-verbal, dépôts, règlement de l'ordre du jour) reviennent à chaque séance.

Export, un fichier par chambre : `output/timeline/arbre_rubriques_chambre.csv` et `arbre_rubriques_senat.csv`.

In [ ]:
def fold(title: str) -> str:
    # Clé de regroupement : sans accents, casse ni ponctuation finale.
    text = unicodedata.normalize("NFD", title)
    text = "".join(c for c in text if not unicodedata.combining(c))
    return " ".join(text.casefold().rstrip(" .:;,").split())


ordered = rubrics.sort_values(["chambre", "date", "fichier", "séance", "rang"], kind="stable")
ordered = ordered.assign(
    position=ordered.groupby("chambre").cumcount(),
    niveau=ordered["parent"].isna().map({True: 1, False: 2}),
    clé=ordered["rubrique"].map(fold),
    clé_parent=ordered["parent"].fillna(ordered["rubrique"]).map(fold),
    jour=ordered["date"].dt.strftime("%d/%m"),
)

tree = (
    ordered.groupby(["chambre", "clé_parent", "niveau", "clé"])
    .agg(
        rubrique=("rubrique", lambda s: s.mode().iloc[0]),  # graphie la plus fréquente
        parent=("parent", lambda s: s.dropna().mode().iloc[0] if s.notna().any() else None),
        occurrences=("rubrique", "size"),
        séances=("fichier", lambda s: s.str.cat(ordered.loc[s.index, "séance"].astype(str)).nunique()),
        interventions=("interventions", "sum"),
        dates=("jour", lambda s: ", ".join(dict.fromkeys(s))),
        première=("position", "min"),
    )
    .reset_index()
)
# Un enfant suit son parent ; un parent sans ligne de niveau 1 (TITL hors séance) garde la position de l'enfant.
parent_first = tree[tree["niveau"] == 1].set_index(["chambre", "clé"])["première"]
tree["ordre_parent"] = [
    parent_first.get((c, k), p) for c, k, p in zip(tree["chambre"], tree["clé_parent"], tree["première"])
]
tree = tree.sort_values(["chambre", "ordre_parent", "clé_parent", "niveau", "première"]).reset_index(drop=True)

columns = ["niveau", "rubrique", "parent", "occurrences", "séances", "interventions", "dates"]
out_dir = OUTPUT_DIR / "timeline"
out_dir.mkdir(parents=True, exist_ok=True)
for chamber, part in tree.groupby("chambre"):
    part[columns].to_csv(out_dir / f"arbre_rubriques_{chamber}.csv", index=False)
    view = part.assign(rubrique=part["rubrique"].where(part["niveau"] == 1, "    └ " + part["rubrique"]))
    print(f"{CHAMBER_NAMES[chamber]} : {(part['niveau'] == 1).sum()} rubriques, {(part['niveau'] == 2).sum()} sous-rubriques")
    display(
        view[["rubrique", "occurrences", "séances", "interventions", "dates"]]
        .style.hide(axis="index")
        .set_properties(**{"text-align": "left", "white-space": "pre"})
    )

## Timeline

Une colonne par séance (dans l'ordre chronologique ; plusieurs séances le même jour sont côte à côte), les rubriques de haut en bas dans l'ordre de la séance. La taille du point suit le nombre d'interventions (point creux : aucun orateur). Le survol donne la liste des orateurs.

Dans le HTML exporté, un champ de recherche (insensible à la casse et aux accents) laisse en couleur les rubriques dont le titre contient le texte saisi et grise les autres.

In [ ]:
MONTHS = ["janv.", "févr.", "mars", "avr.", "mai", "juin", "juil.", "août", "sept.", "oct.", "nov.", "déc."]
COLOR = "#2a78d6"  # slot 1 de la palette catégorielle validée (outil dataviz)
MUTED_COLOR = "#d3d2cd"  # rubriques hors recherche
OPACITY, MUTED_OPACITY = 0.85, 0.35
MAX_LISTED_SPEAKERS = 25


def ordinal(n: int) -> str:
    return "1re" if n == 1 else f"{n}e"


def sitting_label(row) -> str:
    date = f"{row.date.day} {MONTHS[row.date.month - 1]}"
    return f"{date} ({ordinal(row.séance)})" if row.séances_du_jour > 1 else date


def hover_text(row) -> str:
    lines = [f"<b>{row.rubrique}</b>"]
    if row.parent:
        lines.append(f"<i>dans : {row.parent}</i>")
    sitting = f"séance n° {int(row.n_séance)}" if pd.notna(row.n_séance) else "séance n° ?"  # numéro imprimé (OCR)
    lines.append(f"{row.date:%d/%m/%Y} — {sitting} — p. {row.page + 1} du fascicule")
    if not row.orateurs:
        lines.append("Aucun orateur")
    else:
        lines.append(f"{row.interventions} interventions, {row.nb_orateurs} orateurs :")
        lines += [f"  {name} ({n})" for name, n in row.orateurs[:MAX_LISTED_SPEAKERS]]
        if row.nb_orateurs > MAX_LISTED_SPEAKERS:
            lines.append(f"  … et {row.nb_orateurs - MAX_LISTED_SPEAKERS} autres")
    return "<br>".join(lines)


def timeline(chamber: str) -> go.Figure:
    df = rubrics[rubrics["chambre"] == chamber].copy()
    df["séances_du_jour"] = df.groupby("fichier")["séance"].transform("nunique")
    df["colonne"] = df.apply(sitting_label, axis=1)
    columns = list(dict.fromkeys(df.sort_values(["date", "séance"], kind="stable")["colonne"]))
    df["hover"] = df.apply(hover_text, axis=1)
    df["taille"] = 8 + 3 * df["interventions"].pow(0.5)

    fig = go.Figure(
        go.Scatter(
            x=df["colonne"],
            y=df["rang"],
            mode="markers",
            text=df["hover"],
            customdata=df["rubrique"],  # cible de la recherche (cf. SEARCH_JS)
            hovertemplate="%{text}<extra></extra>",
            marker={
                "size": df["taille"],
                "color": COLOR,
                # Rubrique sans orateur : point creux.
                "symbol": ["circle-open" if n == 0 else "circle" for n in df["interventions"]],
                "line": {"width": 2, "color": COLOR},
                "opacity": OPACITY,
            },
        )
    )
    fig.update_layout(
        title={
            "text": f"Rubriques de l'ordre du jour — {CHAMBER_NAMES[chamber]}, 1934",
            "yref": "container",
            "y": 1,
            "yanchor": "top",
            "pad": {"t": 20},
        },
        template="plotly_white",
        height=max(600, 22 * (df["rang"].max() + 2)),
        hoverlabel={"align": "left"},
        margin={"l": 60, "r": 20, "t": 70, "b": 90},
    )
    fig.update_xaxes(categoryorder="array", categoryarray=columns, tickangle=-45, title=None)
    fig.update_yaxes(autorange="reversed", title="ordre dans la séance", zeroline=False)
    return fig


# Recherche dans les titres, injectée dans le HTML exporté (`{plot_id}` est remplacé par plotly).
SEARCH_JS = r'''
const gd = document.getElementById("{plot_id}");
const fold = (s) => s.normalize("NFD").replace(/[\u0300-\u036f]/g, "").toLowerCase();
const titles = gd.data[0].customdata.map(fold);
const bar = document.createElement("div");
bar.style.cssText = "font: 14px sans-serif; margin: 12px 60px; display: flex; gap: 12px; align-items: center;";
bar.innerHTML = '<input type="search" placeholder="Rechercher dans les titres…" style="font: inherit; padding: 6px 10px; width: 320px;"><span></span>';
gd.parentNode.insertBefore(bar, gd);
const [input, count] = bar.children;
input.addEventListener("input", () => {
  const q = fold(input.value.trim());
  const hit = titles.map((t) => !q || t.includes(q));
  const colors = hit.map((h) => (h ? "%(color)s" : "%(muted)s"));
  Plotly.restyle(gd, {
    "marker.color": [colors],
    "marker.line.color": [colors],
    "marker.opacity": [hit.map((h) => (h ? %(opacity)s : %(muted_opacity)s))],
  });
  count.textContent = q ? `${hit.filter(Boolean).length} / ${titles.length} rubriques` : "";
});
''' % {"color": COLOR, "muted": MUTED_COLOR, "opacity": OPACITY, "muted_opacity": MUTED_OPACITY}

### Chambre des députés

In [ ]:
fig_chambre = timeline("chambre")
fig_chambre.show()

### Sénat

In [ ]:
fig_senat = timeline("senat")
fig_senat.show()

## Écriture

Timelines en HTML autonome → `output/timeline/`.

In [ ]:
for chamber, fig in [("chambre", fig_chambre), ("senat", fig_senat)]:
    fig.write_html(out_dir / f"timeline_{chamber}.html", include_plotlyjs="cdn", post_script=SEARCH_JS)
print("→", out_dir)